In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import xgboost as xgb

from catboost import CatBoostClassifier



In [2]:
X_train = pd.read_csv('data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('data/sample_submission.csv',index_col='ROW_ID')

# Features

In [3]:
ret_cols = [c for c in X_test.columns if c.startswith("RET_")]
vol_cols = [c for c in X_test.columns if c.startswith("SIGNED_VOLUME_")]

def fillna_row_mean(df, cols):
    A = df[cols].to_numpy(dtype=float)
    m = np.isnan(A)
    if m.any():
        row_mean = np.nanmean(A, axis=1)
        # si une ligne est full-NaN (rare), remplace la "moyenne" NaN par 0
        row_mean = np.where(np.isfinite(row_mean), row_mean, 0.0)
        r, c = np.where(m)
        A[r, c] = row_mean[r]
        df.loc[:, cols] = A
    return df

X_test = fillna_row_mean(X_test, ret_cols)
X_test = fillna_row_mean(X_test, vol_cols)

In [4]:
RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']

In [5]:
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i+1]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[ f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i+1]].mean(1)
    X_test[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

In [6]:
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]

EMA

In [7]:
ret_cols = [f"RET_{i}" for i in range(1, 21)]
r = X_train[ret_cols].to_numpy()

rtest = X_test[ret_cols].to_numpy()

def ema(arr, L):
    alpha = 2/(L+1)
    w = (1-alpha) ** np.arange(L)  # 0..L-1
    w = w / w.sum()
    return (arr[:, :L] * w).sum(axis=1)

X_train["ema3"]  = ema(r, 3)
X_train["ema5"]  = ema(r, 5)
X_train["ema10"] = ema(r,10)


X_test["ema3"]  = ema(rtest, 3)
X_test["ema5"]  = ema(rtest, 5)
X_test["ema10"] = ema(rtest,10)

m20 = r.mean(axis=1)
s20 = r.std(axis=1, ddof=0)

m20test = rtest.mean(axis=1)
s20test = rtest.std(axis=1, ddof=0)

X_train["z20"] = m20 / (s20 + 1e-12)
X_test["z20"] = m20test / (s20test + 1e-12)

Streak de signe et entropie 

In [8]:
sign = np.sign(r)  
signtest = np.sign(rtest)

def last_streak_len(sig_row, positive=True):
    # part de RET_1 vers RET_20
    target = 1 if positive else -1
    cnt = 0
    for v in sig_row[:20]:  # [:20] explicite
        if v == target:
            cnt += 1
        else:
            break
    return cnt

X_train["streak_pos"] = [last_streak_len(s, True) for s in sign]
X_train["streak_neg"] = [last_streak_len(s, False) for s in sign]

X_test["streak_pos"] = [last_streak_len(s, True) for s in signtest]
X_test["streak_neg"] = [last_streak_len(s, False) for s in signtest]

p = (r > 0).mean(axis=1)
p = np.clip(p, 1e-9, 1 - 1e-9)
X_train["sign_entropy20"] = -(p*np.log(p) + (1-p)*np.log(1-p))

ptest = (rtest > 0).mean(axis=1)    
ptest = np.clip(ptest, 1e-9, 1 - 1e-9)
X_test["sign_entropy20"] = -(ptest*np.log(ptest) + (1-ptest)*np.log(1-ptest))


Qualité de rendement (Sharpe10, Sortino10, t-stat(mean10))

In [9]:
r10 = r[:, :10]
m10 = r10.mean(axis=1)
s10 = r10.std(axis=1, ddof=0)
neg10 = np.minimum(r10, 0)

r10test = rtest[:, :10]
m10test = r10test.mean(axis=1)
s10test = r10test.std(axis=1, ddof=0)
neg10test = np.minimum(r10test, 0)

X_train["sharpe10"]   = m10 / (s10 + 1e-12)
downside10 = np.sqrt((neg10**2).mean(axis=1))
X_train["sortino10"]  = m10 / (downside10 + 1e-12)
X_train["tstat_mean10"] = m10 / (s10/np.sqrt(10) + 1e-12)

X_test["sharpe10"]   = m10test / (s10test + 1e-12)
downside10test = np.sqrt((neg10test**2).mean(axis=1))
X_test["sortino10"]  = m10test / (downside10test + 1e-12)
X_test["tstat_mean10"] = m10test / (s10test/np.sqrt(10) + 1e-12)


In [10]:
X_train["vol10"]         = s10
X_train["downside_dev10"]= downside10

X_test["vol10"]         = s10test
X_test["downside_dev10"]= downside10test

def maxdd_and_recovery(row):
    c = np.cumsum(row)                   # equity curve 20j
    peak = np.maximum.accumulate(c)
    dd = (c - peak)
    maxdd = dd.min()                     # drawdown (négatif)
    # recovery: jours depuis le dernier pic
    last_peak_idx = np.where(c == peak)[0][-1]
    recovery = 20 - 1 - last_peak_idx
    return maxdd, recovery

md_rec = np.apply_along_axis(maxdd_and_recovery, 1, r)
X_train["maxdd20"]   = md_rec[:,0]
X_train["recovery20"]= md_rec[:,1]

md_rec_test = np.apply_along_axis(maxdd_and_recovery, 1, rtest)
X_test["maxdd20"]    = md_rec_test[:,0]
X_test["recovery20"] = md_rec_test[:,1]


Liquidity / turnover (moyenne/écart-type volumes, autocorr, corr ret–liq ; turnover moyen)

In [11]:

ret_cols = [f"RET_{i}" for i in range(1, 21)]
vol_cols = [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]
EPS = 1e-12

def add_volume_features(X):
    v = np.nan_to_num(X[vol_cols].to_numpy(), nan=0.0)
    r = np.nan_to_num(X[ret_cols].to_numpy(), nan=0.0)

    X["sv_mean20"] = v.mean(axis=1)
    X["sv_std20"]  = v.std(axis=1, ddof=0)

    v_mean = v.mean(axis=1, keepdims=True)
    vx = v - v_mean
    num = (vx[:,1:] * vx[:,:-1]).sum(axis=1)
    den = np.sqrt((vx[:,1:]**2).sum(axis=1) * (vx[:,:-1]**2).sum(axis=1))
    X["sv_autocorr1"] = num / (den + EPS)

    rx = r - r.mean(axis=1, keepdims=True)
    num = (rx * vx).sum(axis=1)
    den = np.sqrt((rx**2).sum(axis=1) * (vx**2).sum(axis=1))
    X["corr_ret_sv20"] = num / (den + EPS)

    X["avg_turnover"] = X["AVG_DAILY_TURNOVER"]
    return X

X_train = add_volume_features(X_train.copy())
X_test  = add_volume_features(X_test.copy())


In [12]:
2773 * 65

180245

In [13]:
features = [col for col in X_train.columns if col not in ['TS', 'ALLOCATION']]

In [115]:
len(features)

63

# Preprocessing

In [66]:
import time
import logging
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from typing import Iterable, Tuple, Dict, List, Optional
from sklearn.model_selection import GroupKFold
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster

logger = logging.getLogger("RobustFeatureSelector")
# Exemple de config de base (à placer une seule fois dans ton script principal)
# logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

class RobustFeatureSelector:
    """
    Sélection robuste pour low-signal:
      1) Pré-filtre (quasi-constantes, duplicats)
      2) Ranking CV (Spearman + MI)
      3) Pruning colinéarité via clustering sur |corr|
      4) Stability selection par LassoCV répété

    Fit par fold (groups=TS). Transform standardise si demandé.
    """
    def __init__(
        self,
        groups: Iterable,
        k_folds: int = 10,
        const_thresh: float = 1e-6,
        dup_check: bool = True,
        col_thresh: float = 0.90,
        lasso_seeds: Tuple[int, ...] = (41, 42, 43, 44, 45),
        lasso_cv: int = 5,
        lasso_alphas: int = 50,
        stability_q: float = 0.6,
        max_keep: Optional[int] = None,
        standardize: bool = True,
        random_state: int = 42,
    ):
        self.groups = np.asarray(list(groups))
        self.k_folds = k_folds
        self.const_thresh = const_thresh
        self.dup_check = dup_check
        self.col_thresh = col_thresh
        self.lasso_seeds = lasso_seeds
        self.lasso_cv = lasso_cv
        self.lasso_alphas = lasso_alphas
        self.stability_q = stability_q
        self.max_keep = max_keep
        self.standardize = standardize
        self.random_state = random_state

        self.fast_spearman = True         # new
        self.mi_mode = "none"             # {"none","subsample"} new
        self.mi_subsample_rows = 20000    
        # fitted
        self.keep_cols_: Optional[List[str]] = None
        self.scores_: Optional[pd.DataFrame] = None
        self.rep_map_: Optional[Dict[str, str]] = None
        self.scaler_: Optional[StandardScaler] = RobustScaler()

    # ------------------------- utils internes -------------------------

    def _timeit(self, msg: str):
        class T:
            def __init__(self, message): self.message, self.t0 = message, time.time()
            def __enter__(self): print(self.message); return self
            def __exit__(self, exc_type, exc, tb):
                print(f"{self.message} — done in {time.time()-self.t0:.2f}s")
        return T(msg)

    def _prefilter(self, X: pd.DataFrame) -> pd.DataFrame:
        with self._timeit("[Prefilter] Start"):
            n0 = X.shape[1]
            var = X.var(axis=0, ddof=0)
            keep = var[var > self.const_thresh].index.tolist()
            X = X[keep]
            n1 = X.shape[1]
            print(f"[Prefilter] Kept {n1}/{n0} (drop quasi-const)")

            if self.dup_check:
                uniq, seen = [], {}
                for c in X.columns:
                    sig = (float(X[c].mean()), float(X[c].std(ddof=0)), float(X[c].skew()))
                    if sig in seen: continue
                    seen[sig] = c
                    uniq.append(c)
                X = X[uniq]
                print(f"[Prefilter] After duplicate-like signature: {X.shape[1]} cols")
        return X

    def _cv_scores(self, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        with self._timeit("[Ranking] Fast Spearman/MI"):
            # ensure 1D y
            yv = y #y.values.ravel().astype(np.float32, copy=False)
            # cast X to float32 for speed/memory
            Xf = X.astype(np.float32, copy=False)

            # Precompute ranks ONCE globally (O(n*p)) — then slice by fold
            # Spearman(x,y) = Pearson(rank(x), rank(y))
            if self.fast_spearman:
                # rank each column (ties averaged)
                # Using argsort twice trick to get ranks fast
                A = Xf.to_numpy()
                n, p = A.shape
                # ranks per column
                order = np.argsort(A, axis=0)
                ranks = np.empty_like(order, dtype=np.float32)
                # average ranks for ties: approximate via first rank (fast); good enough in practice
                # (If you need exact average ranks for many ties, use scipy.stats.rankdata column-wise.)
                for j in range(p):
                    ranks[order[:, j], j] = np.arange(1, n+1, dtype=np.float32)
                # y ranks
                y_order = np.argsort(yv)
                y_ranks = np.empty_like(y_order, dtype=np.float32)
                y_ranks[y_order] = np.arange(1, n+1, dtype=np.float32)
            else:
                # Fallback: use Pearson directly (not recommended)
                ranks = Xf.to_numpy()
                y_ranks = yv

            gkf = GroupKFold(n_splits=self.k_folds)
            rho_sum = pd.Series(0.0, index=X.columns)
            mi_sum  = pd.Series(0.0, index=X.columns)
            cnt     = pd.Series(0,    index=X.columns, dtype=int)

            for fi, (tr, va) in enumerate(gkf.split(X, y, groups=self.groups), 1):
                Rtr = ranks[tr, :]          # (n_tr, p)
                yr  = y_ranks[tr]           # (n_tr,)

                # Standardize ranks in the fold, vectorized
                Rtr_m = Rtr.mean(axis=0, dtype=np.float64)  # float64 for numeric stability
                Rtr_s = Rtr.std(axis=0, ddof=0, dtype=np.float64)
                Rtr_s[Rtr_s == 0] = 1.0
                Zx = (Rtr - Rtr_m) / Rtr_s

                ym = yr.mean(dtype=np.float64)
                ys = yr.std(ddof=0, dtype=np.float64)
                ys = ys if ys != 0 else 1.0
                zy = (yr - ym) / ys

                # Pearson on ranks via one matmul: corr_j = <Zx_j, zy> / (n_tr-1)
                num = Zx.T @ zy
                den = (len(tr) - 1)
                rho_fold = np.abs(num / (den if den > 0 else 1)).astype(np.float64)

                rho_sum += pd.Series(rho_fold, index=X.columns)
                cnt     += 1

                # MI: optional & cheap — compute once on a subsample of TRAIN (not per fold)
                if self.mi_mode == "subsample" and fi == 1:
                    # Subsample rows for MI
                    idx_sub = tr if len(tr) <= self.mi_subsample_rows else np.random.RandomState(self.random_state).choice(tr, size=self.mi_subsample_rows, replace=False)
                    # Use discretized bins on ranks (fast)
                    #  e.g., 20 bins uniform on rank scale
                    bins = 20
                    xr = ranks[idx_sub, :]
                    yr_sub = y_ranks[idx_sub]
                    # vectorized, approximate MI by average of per-feature MI via histogram2d
                    mi_vals = np.zeros(p, dtype=np.float64)
                    # Precompute y hist terms
                    y_hist, _ = np.histogram(yr_sub, bins=bins, range=(1, len(yv)))
                    py = y_hist / y_hist.sum()
                    log_py = np.log(py + 1e-12)
                    Hy = -(py * log_py).sum()
                    for j in range(p):
                        x_hist, _ = np.histogram(xr[:, j], bins=bins, range=(1, len(yv)))
                        px = x_hist / x_hist.sum()
                        Hx = -(px * np.log(px + 1e-12)).sum()
                        Hxy, _, _ = np.histogram2d(xr[:, j], yr_sub, bins=bins, range=[[1, len(yv)], [1, len(yv)]])
                        Pxy = Hxy / Hxy.sum()
                        Hxy = -(Pxy * np.log(Pxy + 1e-12)).sum()
                        mi_vals[j] = max(0.0, Hx + Hy - Hxy)
                    mi_sum += pd.Series(mi_vals, index=X.columns)

                # log each fold quickly
                print(f"[Ranking-fast] Fold {fi}: computed Spearman in vectorized mode")

            rho_mean = rho_sum / cnt.clip(lower=1)
            if self.mi_mode == "subsample":
                mi_mean = mi_sum / 1.0  # computed once
            else:
                # No MI → zeros
                mi_mean = pd.Series(0.0, index=X.columns)

            # normalize & aggregate
            r = (rho_mean - rho_mean.min()) / (rho_mean.max() - rho_mean.min() + 1e-12)
            m = (mi_mean  - mi_mean.min())  / (mi_mean.max()  - mi_mean.min()  + 1e-12) if self.mi_mode != "none" else 0.0
            score = 0.8*r + 0.2*m if self.mi_mode != "none" else r  # weight Spearman more

            out = pd.DataFrame({"spearman": rho_mean, "mi": mi_mean, "score": score})
            print(f"[Ranking-fast] Done. Top5:\n{out.sort_values('score', ascending=False).head(5)}")
            return out

    def _prune_collinearity(self, X: pd.DataFrame, score: pd.DataFrame):
        with self._timeit("[Prune] Collinearity clustering (robust)"):
            # Corr absolue, nettoyage
            C = X.corr().abs().replace([np.inf, -np.inf], np.nan).fillna(0.0)

            # Symétrise + clippe strictement dans [0,1]
            C = (C + C.T) / 2.0
            Cv = C.to_numpy(dtype=np.float64, copy=True)
            np.fill_diagonal(Cv, 1.0)
            np.clip(Cv, 0.0, 1.0, out=Cv)

            # Distance = 1 - corr, zéro sur la diagonale
            D = 1.0 - Cv
            np.fill_diagonal(D, 0.0)

            # Nettoyage num. final: symétrie + coupe les valeurs <0 résiduelles
            D = (D + D.T) / 2.0
            D[D < 0] = 0.0

            # Cas dégénéré: si tout est 0 (features identiques), force un seul cluster
            if np.allclose(D, 0.0):
                keep = [score["score"].idxmax()]
                rep_map = {c: keep[0] for c in C.columns}
                return keep, rep_map

            # Vers format condensé sans re-vérif (on a déjà nettoyé)
            from scipy.spatial.distance import squareform
            from scipy.cluster.hierarchy import linkage, fcluster

            dvec = squareform(D, checks=False)
            Z = linkage(dvec, method="average", optimal_ordering=False)

            labels = fcluster(Z, t=1 - self.col_thresh, criterion="distance")
            lab = pd.Series(labels, index=C.columns)

            keep, rep_map = [], {}
            for k in np.unique(labels):
                cols = lab.index[lab == k]
                best = score.loc[cols, "score"].idxmax()
                keep.append(best)
                rep_map.update({c: best for c in cols})

            keep = sorted(set(keep))
            print(f"[Prune] Clusters: {len(np.unique(labels))} | Kept reps: {len(keep)}")
            return keep, rep_map

    def _stability_lasso(self, X: pd.DataFrame, y: pd.Series, cols: List[str]) -> Tuple[List[str], pd.Series]:
        """
        Stability selection basée sur RidgeCV (pas de sparsité dure) :
        - Standardise X (optionnel)
        - RidgeCV avec grille d'alphas logspace
        - Sélectionne les TOP-K coefficients par |coef| à chaque run (seed x fold)
        - Garde les features dont la fréquence >= stability_q
        - Fallback: top-N par fréquence si trop strict
        Retourne (liste_features, fréquence_selection_par_feature).
        """

        with self._timeit("[Stability] RidgeCV (TOP-K by |coef|)"):
            if len(cols) == 0:
                logger.warning("[Stability] No columns provided.")
                return [], pd.Series(dtype=float)

            # Grille d'alphas "safe" low-signal
            ALPHAS = np.logspace(2, 20, 15)
            CVFOLDS = max(3, min(5, self.lasso_cv))  # 3..5
            # Choix TOP-K: ~15% des features avec plancher 40
            TOPK = max(40, int(0.15 * len(cols)))

            sel_counts = pd.Series(0, index=cols, dtype=int)
            gkf = GroupKFold(n_splits=self.lasso_cv)
            total = 0

            # y en 1D
            y_arr = np.asarray(y).ravel()

            for seed in self.lasso_seeds:
                for fi, (tr, va) in enumerate(gkf.split(X[cols], y_arr, groups=self.groups), 1):
                    Xtr_df = X.iloc[tr][cols].astype(float)
                    ytr = y_arr[tr]

                    # >>> scaling robuste + nettoyage <<<
                    Xtr_ = self._safe_scale_matrix(Xtr_df, q_clip=(0.001, 0.999))

                    ridge = RidgeCV(
                        alphas=ALPHAS, cv=CVFOLDS,
                        scoring="neg_mean_squared_error", fit_intercept=True
                    ).fit(Xtr_, ytr)

                    coef = pd.Series(np.abs(ridge.coef_), index=cols)
                    keep_run = coef.sort_values(ascending=False).head(min(TOPK, len(cols))).index
                    sel_counts.loc[keep_run] += 1
                    total += 1
                    
            freq = sel_counts / max(total, 1)
            keep = freq[freq >= self.stability_q].index.tolist()

            if len(keep) == 0:
                # Fallback: compléter par les meilleures fréquences (jusqu'à 80 ou tout)
                cap = min(max(80, TOPK), len(cols))
                keep = freq.sort_values(ascending=False).head(cap).index.tolist()
                print(f"[Stability] No feat >= q={self.stability_q:.2f}; fallback to top-{cap} by frequency.")

            print(f"[Stability] Kept {len(keep)} | median freq={freq.median():.2f} | total runs={total} | TOPK/run={TOPK}")
            return keep, freq


    # ------------------------------ API ------------------------------

    def fit(self, X: pd.DataFrame, y: pd.Series):
        t0 = time.time()
        print(f"[Fit] Start on X={X.shape}, y={y.shape}, k_folds={self.k_folds}")
        X0 = self._prefilter(X)

        scores = self._cv_scores(X0, y)

        if self.max_keep:
            top = scores.sort_values("score", ascending=False).head(self.max_keep).index.tolist()
            X1  = X0[top]
            scores = scores.loc[top]
            print(f"[Fit] max_keep={self.max_keep} → working set = {X1.shape[1]}")
        else:
            X1 = X0

        kept_after_cluster, rep_map = self._prune_collinearity(X1, scores)
        kept_after_lasso, freq = self._stability_lasso(X1, y, kept_after_cluster)

        self.keep_cols_ = kept_after_lasso
        self.scores_ = scores.join(freq.rename("stability_freq"), how="left")
        self.rep_map_ = rep_map

        if self.standardize and len(self.keep_cols_) > 0:
            self.scaler_ = self._safe_scale_matrix(X[self.keep_cols_])
            print(f"[Fit] Scaler fitted on {len(self.keep_cols_)} features.")

        print(f"[Fit] Done. Kept={len(self.keep_cols_)} features in {time.time()-t0:.2f}s")
        
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.keep_cols_ is None:
            raise RuntimeError("Call fit() before transform().")
        with self._timeit("[Transform]"):
            missing = [c for c in self.keep_cols_ if c not in X.columns]
            if missing:
                print(f"[Transform] Missing {len(missing)} columns; filling with zeros.")
            Xs = pd.DataFrame(index=X.index)
            for c in self.keep_cols_:
                if c in X.columns:
                    Xs[c] = X[c].astype(float)
                else:
                    Xs[c] = 0.0
            # if self.standardize and self.scaler_ is not None:
            #     Xs.loc[:, :] = self.scaler_.transform(Xs)
            print(f"[Transform] Output shape={Xs.shape}")
        return Xs

    def fit_transform(self, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        return self.fit(X, y).transform(X)

    def _safe_scale_matrix(self, X: pd.DataFrame, q_clip=(0.001, 0.999)) -> np.ndarray:
        A = X.to_numpy(dtype=np.float64, copy=True)
        # clip par quantiles par colonne (anti-outliers)
        if q_clip is not None:
            lo = np.nanquantile(A, q_clip[0], axis=0)
            hi = np.nanquantile(A, q_clip[1], axis=0)
            A = np.minimum(np.maximum(A, lo), hi)
        # standardisation robuste
        mu = np.nanmean(A, axis=0)
        sd = np.nanstd(A, axis=0, ddof=0)
        sd[~np.isfinite(sd)] = 1.0
        sd[sd < 1e-12] = 1.0  # évite division par 0
        A = (A - mu) / sd
        # sanitize final
        A[~np.isfinite(A)] = 0.0
        # borne pour éviter overflow matmul
        np.clip(A, -50.0, 50.0, out=A)
        return A

    def _sanitize_vector(self, y: pd.Series) -> np.ndarray:
        v = np.asarray(y).ravel().astype(np.float64, copy=False)
        v[~np.isfinite(v)] = 0.0
        return v


In [86]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

n_splits = 10
#kf = KFold(n_splits=n_splits, shuffle=True, random_state=43)


groups = X_train["TS"]  

train_dates = X_train['TS'].unique()
scores = []
models = []
best_rounds = []
best_scores = []

y_train_bin = y_train.copy()
y_train_bin["target"] = (y_train_bin["target"] > 0).astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
(tr_idx_all, hold_idx) = next(gss.split(X_train, y_train, groups))
X_cv, y_cv = X_train.iloc[tr_idx_all].reset_index(drop=True), y_train.iloc[tr_idx_all].reset_index(drop=True)
X_hold, y_hold = X_train.iloc[hold_idx].reset_index(drop=True), y_train.iloc[hold_idx].reset_index(drop=True)

# OOF et HOLDOUT preds containers
oof_pred = np.full(len(X_train), np.nan)
oof_cls  = np.zeros(len(X_train), dtype=int)
fold_acc = []
fold_models = []
fold_selectors = []
fold_thresholds = []
hold_preds = []  


hold_votes = np.zeros((n_splits, len(X_hold)), dtype=int)
test_votes = np.zeros((n_splits, len(X_test)), dtype=int)


gkf = GroupKFold(n_splits=n_splits)


In [76]:
X_hold.columns

Index(['TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17', 'RET_16',
       'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10', 'RET_9',
       'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2', 'RET_1',
       'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'AVG_DAILY_TURNOVER',
       'AVERAGE_PERF_3', 'ALLOCATIONS_AVERAGE_PERF_3', 'AVERAGE_PERF_5',
       'ALLOCATIONS_AVERAGE_PERF_5', 'AVERAGE_PERF_10',
       'ALLOCATIONS_AVERAGE_PERF_10', 'AVERAGE_PERF_15',
       'ALLOCATIONS_AVERAGE_PERF_15', 'AVERAGE_PERF_20',
       'ALLOCATIONS_AVERAGE_PERF_20', 'ema3', 'ema5', 'ema10', 

In [87]:
i = 1 

selected_features = []

ALPHAS = np.logspace(0, 15, 21)

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train["TS"])):

    print(f"[META] new meta fold {i}")
    x_tr, y_tr = X_train.iloc[tr_idx].copy(), y_train.iloc[tr_idx].copy()
    x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx].copy()

    y_val_bin = (y_val["target"] > 0).astype(int)

    x_hold = X_hold.copy()

    # m_tr = X_cv['TS'].isin(tr_dates)
    # m_val = X_cv['TS'].isin(val_dates)

    # x_tr = X_cv.loc[m_tr]
    # y_tr = y_cv.loc[m_tr]
    # x_val = X_cv.loc[m_val]
    # y_val = y_cv.loc[m_val]

    
    # r = 8 
    # x_tr = enrich_with_alloc_ts_embeddings(x_tr, features)
    # x_val = enrich_with_ts_embeddings(x_val, features)
    # alloc_features = [f"alloc_svd_{i+1}" for i in range(r)] + ["alloc_svd_ratio"]

    # alloc_map = x_tr[["ALLOCATION"] + alloc_features].drop_duplicates("ALLOCATION")

    #x_val = x_val.merge(alloc_map, on="ALLOCATION", how="left")

    x_test = X_test.copy()
    
    final_features = [col for col in x_tr.columns if col not in ["TS", "ALLOCATION"]]
    
    selector = RobustFeatureSelector(
    groups=X_train.loc[tr_idx,"TS"],
    k_folds=3,              # was 5–10 → 3 is enough for ranking
    col_thresh=0.95,
    stability_q=0.4,
    max_keep=200,
    standardize=True,)

    selector.fast_spearman = True
    selector.mi_subsample_rows = 20000                          
        
    # fit_kwargs['sample_weight'] = sw
    with np.errstate(all='ignore'):
        x_tr = selector.fit_transform(x_tr[final_features], np.asarray(y_tr).ravel())
        x_val = selector.transform(x_val[final_features])

        x_test = selector.transform(x_test[final_features])

        selected_features.append(list(x_test.columns))
        #X_hold_trans = selector.transform(x_hold)
        
        ridge = RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error", fit_intercept=True)
        ridge.fit(x_tr, y_tr)


    # ---- prédictions ----
    p_val = ridge.predict(x_val)       # régression (réels)
    # on choisit un seuil t* qui maximise l’accuracy sur la val
    cand = np.unique(np.quantile(p_val, np.linspace(0.05, 0.95, 31)))  # petite grille
    cand = np.append(cand, 0.0)  # inclure le 0 au cas où
    accs = [(thr, accuracy_score(y_val_bin, (p_val >= thr).astype(int))) for thr in cand]
    t_star, acc_star = max(accs, key=lambda x: x[1])


    oof_pred[val_idx] = p_val
    oof_pred[val_idx] = p_val
    oof_cls[val_idx]  = (p_val >= t_star).astype(int)
    fold_acc.append(acc_star)

    # test → vote par fold (utilise le seuil du fold)
    #p_tst = ridge.predict(X_hold_trans)
    #test_votes[i-1, :] = (p_tst >= t_star).astype(int)

    p_tst = ridge.predict(x_test)
    test_votes[i-1, :] = (p_tst >= t_star).astype(int)

    # stock
    fold_models.append(ridge)
    fold_selectors.append(selector)
    fold_thresholds.append(t_star)

    print(f"[META] Fold {i:02d} | kept={x_tr.shape[1]} | alpha*={ridge.alpha_:.4g} | thr*={t_star:.4g} | Acc(val)={acc_star*100:.2f}%")    

    i+= 1 

oof_acc = accuracy_score(y_train_bin, oof_cls)
print(f"\nOOF Accuracy = {oof_acc*100:.2f}% | mean(fold Acc) = {np.mean(fold_acc)*100:.2f}% ± {np.std(fold_acc)*100:.2f}%")

# ---- inférence sur X_test (vote majoritaire des folds) ----
#hold_vote_mean = test_votes.mean(axis=0)           # fraction de folds qui votent 1
#y_hold_hat_cls = (hold_vote_mean >= 0.5).astype(int)


[META] new meta fold 1
[Fit] Start on X=(162175, 70), y=(162175,), k_folds=3
[Prefilter] Start
[Prefilter] Kept 63/70 (drop quasi-const)
[Prefilter] After duplicate-like signature: 62 cols
[Prefilter] Start — done in 0.33s
[Ranking] Fast Spearman/MI
[Ranking-fast] Fold 1: computed Spearman in vectorized mode
[Ranking-fast] Fold 2: computed Spearman in vectorized mode
[Ranking-fast] Fold 3: computed Spearman in vectorized mode
[Ranking-fast] Done. Top5:
            spearman   mi     score
RET_1       0.051638  0.0  1.000000
ema3        0.048985  0.0  0.946810
ema5        0.043656  0.0  0.839943
ema10       0.039398  0.0  0.754557
streak_neg  0.038019  0.0  0.726895
[Ranking] Fast Spearman/MI — done in 1.88s
[Fit] max_keep=200 → working set = 62
[Prune] Collinearity clustering (robust)
[Prune] Clusters: 61 | Kept reps: 61
[Prune] Collinearity clustering (robust) — done in 1.00s
[Stability] RidgeCV (TOP-K by |coef|)
[Stability] Kept 51 | median freq=0.80 | total runs=25 | TOPK/run=40
[Sta

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.97598e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.68928e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.52122e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Li

[META] Fold 01 | kept=51 | alpha*=1 | thr*=3.599e-05 | Acc(val)=52.83%
[META] new meta fold 2
[Fit] Start on X=(162175, 70), y=(162175,), k_folds=3
[Prefilter] Start
[Prefilter] Kept 63/70 (drop quasi-const)
[Prefilter] After duplicate-like signature: 62 cols
[Prefilter] Start — done in 0.23s
[Ranking] Fast Spearman/MI
[Ranking-fast] Fold 1: computed Spearman in vectorized mode
[Ranking-fast] Fold 2: computed Spearman in vectorized mode
[Ranking-fast] Fold 3: computed Spearman in vectorized mode
[Ranking-fast] Done. Top5:
            spearman   mi     score
RET_1       0.050576  0.0  1.000000
ema3        0.046577  0.0  0.918183
ema5        0.042668  0.0  0.838202
ema10       0.038468  0.0  0.752264
streak_neg  0.037778  0.0  0.738152
[Ranking] Fast Spearman/MI — done in 1.84s
[Fit] max_keep=200 → working set = 62
[Prune] Collinearity clustering (robust)
[Prune] Clusters: 61 | Kept reps: 61
[Prune] Collinearity clustering (robust) — done in 1.08s
[Stability] RidgeCV (TOP-K by |coef|)
[S

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.54991e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=4.98712e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=4.90558e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Li

[META] Fold 02 | kept=52 | alpha*=1 | thr*=-2.038e-05 | Acc(val)=52.42%
[META] new meta fold 3
[Fit] Start on X=(162175, 70), y=(162175,), k_folds=3
[Prefilter] Start
[Prefilter] Kept 63/70 (drop quasi-const)
[Prefilter] After duplicate-like signature: 62 cols
[Prefilter] Start — done in 0.25s
[Ranking] Fast Spearman/MI
[Ranking-fast] Fold 1: computed Spearman in vectorized mode
[Ranking-fast] Fold 2: computed Spearman in vectorized mode
[Ranking-fast] Fold 3: computed Spearman in vectorized mode
[Ranking-fast] Done. Top5:
            spearman   mi     score
RET_1       0.052521  0.0  1.000000
ema3        0.047109  0.0  0.891897
ema5        0.041082  0.0  0.771540
ema10       0.036966  0.0  0.689327
streak_pos  0.036866  0.0  0.687340
[Ranking] Fast Spearman/MI — done in 2.02s
[Fit] max_keep=200 → working set = 62
[Prune] Collinearity clustering (robust)
[Prune] Clusters: 61 | Kept reps: 61
[Prune] Collinearity clustering (robust) — done in 1.08s
[Stability] RidgeCV (TOP-K by |coef|)
[

...

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.7888e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.2734e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.16755e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Libr

[META] Fold 09 | kept=49 | alpha*=1 | thr*=2.386e-05 | Acc(val)=51.91%
[META] new meta fold 10
[Fit] Start on X=(162240, 70), y=(162240,), k_folds=3
[Prefilter] Start
[Prefilter] Kept 63/70 (drop quasi-const)
[Prefilter] After duplicate-like signature: 62 cols
[Prefilter] Start — done in 0.21s
[Ranking] Fast Spearman/MI
[Ranking-fast] Fold 1: computed Spearman in vectorized mode
[Ranking-fast] Fold 2: computed Spearman in vectorized mode
[Ranking-fast] Fold 3: computed Spearman in vectorized mode
[Ranking-fast] Done. Top5:
       spearman   mi     score
RET_1  0.052702  0.0  1.000000
ema3   0.048396  0.0  0.917341
ema5   0.042521  0.0  0.804559
z20    0.039463  0.0  0.745846
ema10  0.039055  0.0  0.738028
[Ranking] Fast Spearman/MI — done in 2.27s
[Fit] max_keep=200 → working set = 62
[Prune] Collinearity clustering (robust)
[Prune] Clusters: 61 | Kept reps: 61
[Prune] Collinearity clustering (robust) — done in 1.18s
[Stability] RidgeCV (TOP-K by |coef|)
[Stability] Kept 49 | median fr

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.78268e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.26869e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.09849e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Li

[META] Fold 10 | kept=49 | alpha*=1 | thr*=1.588e-05 | Acc(val)=51.11%

OOF Accuracy = 52.23% | mean(fold Acc) = 52.23% ± 0.47%


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=4.68369e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model

In [88]:
test_votes

array([[0, 1, 0, ..., 1, 0, 0],
       [0, 1, 0, ..., 1, 0, 0],
       [0, 1, 0, ..., 1, 0, 0],
       ...,
       [0, 1, 0, ..., 1, 0, 0],
       [0, 1, 0, ..., 1, 0, 0],
       [0, 1, 0, ..., 1, 0, 0]])

In [96]:
X_test.index

Index([180245, 180246, 180247, 180248, 180249, 180250, 180251, 180252, 180253,
       180254,
       ...
       187970, 187971, 187972, 187973, 187974, 187975, 187976, 187977, 187978,
       187979],
      dtype='int64', name='ROW_ID', length=7735)

In [98]:
# 1) Vote majoritaire par ligne
test_vote_mean = test_votes.mean(axis=0)              # shape: (n_test,)
y_test_hat_cls = (test_vote_mean >= 0.5).astype(int)  # 0/1

# 2) Construire la submission alignée à X_test
assert len(y_test_hat_cls) == len(X_test)
cols_id = [c for c in ["row_id", "ID", "id"] if c in X_test.columns]

if cols_id:  # Cas 1: la compète fournit un identifiant unique
    sub = pd.DataFrame({
        cols_id[0]: X_test[cols_id[0]].values,
        "target": y_test_hat_cls
    })
else:        # Cas 2: pas d'ID → on met (TS, ALLOCATION) ou l'index
    id_cols = [c for c in ["TS", "ALLOCATION"] if c in X_test.columns]
    if id_cols:
        sub = pd.DataFrame({
            **{c: X_test[c].values for c in id_cols},
            "target": y_test_hat_cls
        })
    else:
        sub = pd.DataFrame({"target": y_test_hat_cls}, index=X_test.index)

# (optionnel) ajouter une "proba" = fraction de votes positifs
sub["vote_mean"] = test_vote_mean

sub = sub["target"]
# 3) Tri/sauvegarde
# sub = sub.sort_values(id_cols or [cols_id[0]]).reset_index(drop=True)
sub.to_csv("data/robustfeatureslinear.csv", index=True)
print(sub.head(), "\nSaved -> submission.csv")


0    0
1    1
2    0
3    0
4    0
Name: target, dtype: int64 
Saved -> submission.csv
